In [ ]:
# Deconvolution with anchoring

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# ==========================================
# 1. LOAD & SMOOTH
# ==========================================
file_path = 'spectra_Anod-SS1/a0525vrr.txt'
df = pd.read_csv(file_path, sep='\s+', comment='#', header=None, encoding='latin1')
df = df[(df[0] >= 360) & (df[0] <= 700)]
wn = df[0].values
inty_raw = df[1].values

# Smooth the data
inty_s = savgol_filter(inty_raw, window_length=31, polyorder=3)

# ==========================================
# 2. ANCHOR POINT BG CORRECTION
# ==========================================
# Define regions where there are NO peaks (Adjust based on your real peaks!)
# We use the first 50 units and last 50 units as "anchors"
mask_anchors = ((wn >= 360) & (wn <= 400)) | ((wn >= 650) & (wn <= 700))

wn_anchors = wn[mask_anchors]
inty_anchors = inty_s[mask_anchors]

# Fit polynomial ONLY to these anchor points
poly_coeffs = np.polyfit(wn_anchors, inty_anchors, 2) # 2nd order is usually safer for anchors
poly_bg = np.polyval(poly_coeffs, wn)

# Subtract and Zero-align
inty_corrected = inty_s - poly_bg
inty_corrected -= inty_corrected.min()

# ==========================================
# 3. DECONVOLUTION & PLOTTING
# ==========================================
# [Lorentzian and fit_model functions same as before]
def lorentzian(x, amp, ctr, wid):
    return amp * wid**2 / ((x - ctr)**2 + wid**2)

def fit_model(x, a1, c1, w1, a2, c2, w2):
    return lorentzian(x, a1, c1, w1) + lorentzian(x, a2, c2, w2)

guesses = [max(inty_corrected), 480, 20, max(inty_corrected)/2, 560, 20]
popt, _ = curve_fit(fit_model, wn, inty_corrected, p0=guesses)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(wn, inty_s, color='gray', alpha=0.5, label='Original Smooth')
plt.plot(wn, poly_bg, 'r--', label='Anchor-Fit Background')
plt.scatter(wn_anchors, inty_anchors, color='red', s=10, label='Anchor Points Used')



plt.plot(wn, inty_corrected, 'k', label='Resulting Flat Baseline', lw=1.5)
plt.fill_between(wn, 0, lorentzian(wn, *popt[0:3]), color='blue', alpha=0.3, label='Peak 1')
plt.fill_between(wn, 0, lorentzian(wn, *popt[3:6]), color='green', alpha=0.3, label='Peak 2')

plt.title("Anchor-Point Corrected Deconvolution")
plt.xlabel("Wavenumber ($cm^{-1}$)")
plt.ylabel("Intensity")
plt.legend()
plt.show()

print(f"Peak 1 Center: {popt[1]:.2f}")
print(f"Peak 2 Center: {popt[4]:.2f}")